In [1]:
#A test notebook for sorting out api data aquisition

In [2]:
import sys
print(sys.executable)

/home/djmead/Documents/Projects/Financial/.venv/bin/python


In [4]:
import requests
import json
import os
import keyring, getpass

In [ ]:
service = "bls-api"
user = "DYLAN-MEAD"
api_key = keyring.get_password(service, user)


print(f"API Key: {api_key[:3]}...{api_key[-3:]}")

In [ ]:
def get_bls_timeseries(series_id, start_year, end_year, api_key):
    payload = {
        'seriesid': [series_id],
        'startyear': start_year,
        'endyear': end_year,
        'registrationKey': api_key
    }
    
    headers = {'Content-type': 'application/json'}
    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    response = requests.post(url, data=json.dumps(payload), headers=headers)
    if response.status_code == 200:
        data = response.json()
        if data.get('status') == 'REQUEST_SUCCEEDED':
            return data['Results']['series']
        else:
            raise Exception(f"API request failed: {data.get('message', 'No message provided')}")
    else:
        raise Exception(f"HTTP request failed with status code {response.status_code}")

In [10]:
cpi_id = "CUSR0000SA0"
test_data = get_bls_timeseries(cpi_id, "2025", "2026", api_key)



In [12]:
for item in test_data[0]['data']:
    print(item)

{'year': '2026', 'period': 'M07', 'periodName': 'July', 'latest': 'true', 'value': '332.813', 'footnotes': [{}]}
{'year': '2026', 'period': 'M06', 'periodName': 'June', 'value': '332.568', 'footnotes': [{}]}
{'year': '2026', 'period': 'M05', 'periodName': 'May', 'value': '333.979', 'footnotes': [{}]}
{'year': '2026', 'period': 'M04', 'periodName': 'April', 'value': '332.407', 'footnotes': [{}]}
{'year': '2026', 'period': 'M03', 'periodName': 'March', 'value': '330.293', 'footnotes': [{}]}
{'year': '2026', 'period': 'M02', 'periodName': 'February', 'value': '327.460', 'footnotes': [{}]}
{'year': '2026', 'period': 'M01', 'periodName': 'January', 'value': '326.588', 'footnotes': [{}]}
{'year': '2025', 'period': 'M12', 'periodName': 'December', 'value': '326.031', 'footnotes': [{}]}
{'year': '2025', 'period': 'M11', 'periodName': 'November', 'value': '325.063', 'footnotes': [{}]}
{'year': '2025', 'period': 'M10', 'periodName': 'October', 'value': '-', 'footnotes': [{'code': 'X', 'text': 'D

In [2]:
import requests
import json
import prettytable
headers = {'Content-type': 'application/json'}
data = json.dumps({"seriesid": ['CUUR0000SA0','SUUR0000SA0'],"startyear":"2025", "endyear":"2026"})
p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)
json_data = json.loads(p.text)
for series in json_data['Results']['series']:
    x=prettytable.PrettyTable(["series id","year","period","value","footnotes"])
    seriesId = series['seriesID']
    for item in series['data']:
        year = item['year']
        period = item['period']
        value = item['value']
        footnotes=""
        for footnote in item['footnotes']:
            if footnote:
                footnotes = footnotes + footnote['text'] + ','
        if 'M01' <= period <= 'M12':
            x.add_row([seriesId,year,period,value,footnotes[0:-1]])
    print(x)

+-------------+------+--------+---------+----------------------------------------------------------+
|  series id  | year | period |  value  |                        footnotes                         |
+-------------+------+--------+---------+----------------------------------------------------------+
| CUUR0000SA0 | 2026 |  M07   | 333.918 |                                                          |
| CUUR0000SA0 | 2026 |  M06   | 333.952 |                                                          |
| CUUR0000SA0 | 2026 |  M05   | 335.123 |                                                          |
| CUUR0000SA0 | 2026 |  M04   | 333.020 |                                                          |
| CUUR0000SA0 | 2026 |  M03   | 330.213 |                                                          |
| CUUR0000SA0 | 2026 |  M02   | 326.785 |                                                          |
| CUUR0000SA0 | 2026 |  M01   | 325.252 |                                                  